<a href="https://colab.research.google.com/github/postnicov/ResazurinResorufin/blob/main/Resazurin_Resorufin_Color_Mixing_Reflexion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Resazurin / Resorufin pH-dependent color mixing

This notebook retrieves the pH-dependent absorbance spectra of **resazurin** and **resorufin**, mixes them at user-defined fractions `x` (fraction of resorufin), converts the mixed absorbance spectra into reflectance using the **Kubelka-Munk** theory, and computes the resulting **CIE L\*a\*b\*** and **sRGB** colors. Results are exported to a downloadable Excel file with a colored swatch cell for each pH/x combination.

In [1]:
#@title Install and import required packages
import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = {
    'colour-science': 'colour',
    'pandas': 'pandas',
    'numpy': 'numpy',
    'openpyxl': 'openpyxl',
    'XlsxWriter': 'xlsxwriter',
}

missing_packages = [
    package for package, module in REQUIRED_PACKAGES.items()
    if importlib.util.find_spec(module) is None
]
if missing_packages:
    print('Installing:', ', '.join(missing_packages))
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', *missing_packages])
else:
    print('All required packages are already installed.')

import warnings
from io import BytesIO
from itertools import product

import colour
import numpy as np
import pandas as pd
from colour.utilities import ColourUsageWarning

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

warnings.filterwarnings('ignore', category=ColourUsageWarning)
print('Libraries imported successfully.')

Installing: colour-science, XlsxWriter
Libraries imported successfully.


In [2]:
#@title Set data source URLs and load the spectra tables
RESAZURIN_URL = 'https://raw.githubusercontent.com/postnicov/ResazurinResorufin/refs/heads/main/Data/pHRzSpectra.csv'  #@param {type:"string"}
RESORUFIN_URL = 'https://raw.githubusercontent.com/postnicov/ResazurinResorufin/refs/heads/main/Data/pHRfSpectra.csv'  #@param {type:"string"}

def load_spectra(url):
    """Load a spectra CSV where the first row (after the wavelength header cell) holds pH
    values, the first column holds wavelengths, and the remaining cells are absorbance."""
    raw = pd.read_csv(url, header=0, index_col=0)
    raw.columns = [float(c) for c in raw.columns]
    raw.index = raw.index.astype(float)
    raw = raw.sort_index()
    return raw

resazurin_spectra = load_spectra(RESAZURIN_URL)
resorufin_spectra = load_spectra(RESORUFIN_URL)

common_pH = sorted(set(resazurin_spectra.columns).intersection(resorufin_spectra.columns))
if not common_pH:
    raise ValueError('No common pH values found between the two spectra tables.')

print(f'Resazurin spectrum: {resazurin_spectra.shape[0]} wavelengths x {resazurin_spectra.shape[1]} pH values')
print(f'Resorufin spectrum: {resorufin_spectra.shape[0]} wavelengths x {resorufin_spectra.shape[1]} pH values')
print(f'Common pH values ({len(common_pH)}):', common_pH)

Resazurin spectrum: 341 wavelengths x 17 pH values
Resorufin spectrum: 348 wavelengths x 17 pH values
Common pH values (17): [1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0, 5.5, 6.0, 6.5, 7.0, 7.5, 8.0, 8.5, 9.0]


In [3]:
#@title Set the resorufin fraction range (x)
X_MIN = 0.0  #@param {type:"number"}
X_STEP = 0.1  #@param {type:"number"}
X_MAX = 1.0  #@param {type:"number"}

if X_STEP <= 0:
    raise ValueError('X_STEP must be positive.')
if X_MAX < X_MIN:
    raise ValueError('X_MAX must be greater than or equal to X_MIN.')

n_steps = int(round((X_MAX - X_MIN) / X_STEP))
x_values = [round(X_MIN + i * X_STEP, 10) for i in range(n_steps + 1)]
if x_values[-1] < X_MAX and not np.isclose(x_values[-1], X_MAX):
    x_values.append(round(X_MAX, 10))
x_values = sorted(set(min(max(v, 0.0), 1.0) if abs(v) < 1e-9 or abs(v - 1) < 1e-9 else v for v in x_values))

print(f'Resorufin fraction values x ({len(x_values)}):', x_values)

Resorufin fraction values x (11): [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]


In [4]:
#@title Set the Kubelka-Munk scattering constant S0
S0 = 0.4819  #@param {type:"number"}

print(f'Kubelka-Munk scattering constant S0 = {S0}')

Kubelka-Munk scattering constant S0 = 0.4819


In [5]:
#@title Define spectral mixing, Kubelka-Munk, and colorimetry functions
ILLUMINANT = colour.SDS_ILLUMINANTS['D65']
CMFS = colour.MSDS_CMFS['CIE 1931 2 Degree Standard Observer']
D65_XY = colour.CCS_ILLUMINANTS['CIE 1931 2 Degree Standard Observer']['D65']


def mix_absorbance(wavelengths, abs_rz, abs_rf, x):
    """Linearly mix resazurin and resorufin absorbance spectra: (1-x)*Rz + x*Rf."""
    return (1 - x) * abs_rz + x * abs_rf


def absorbance_to_reflectance(absorbance, S0):
    """Convert absorbance to diffuse reflectance via the Kubelka-Munk relation.

    K/S is taken proportional to absorbance (K = absorbance, S = S0 constant),
    then R = 1 + (K/S) - sqrt((K/S)^2 + 2*(K/S)).
    """
    k_over_s = np.clip(absorbance, 0, None) / S0
    reflectance = 1 + k_over_s - np.sqrt(k_over_s ** 2 + 2 * k_over_s)
    return np.clip(reflectance, 0.0, 1.0)


def spectrum_to_lab_rgb(wavelengths, reflectance):
    """Convert a reflectance spectrum (0-1) sampled at given wavelengths to CIE Lab and sRGB."""
    sd = colour.SpectralDistribution(
        dict(zip(wavelengths, reflectance))
    ).align(colour.SpectralShape(int(wavelengths.min()), int(wavelengths.max()), 1))
    xyz = colour.sd_to_XYZ(sd, cmfs=CMFS, illuminant=ILLUMINANT) / 100.0
    lab = colour.XYZ_to_Lab(xyz, D65_XY)
    rgb = colour.XYZ_to_sRGB(xyz)
    rgb_8bit = np.clip(np.round(rgb * 255), 0, 255).astype(int)
    return lab, rgb_8bit


print('Core functions defined.')

Core functions defined.


In [6]:
#@title Compute mixed spectra and colors for all pH and x combinations
wavelengths = resazurin_spectra.index.to_numpy(dtype=float)
wavelengths_rf = resorufin_spectra.index.to_numpy(dtype=float)

if not np.array_equal(wavelengths, wavelengths_rf):
    common_wavelengths = np.intersect1d(wavelengths, wavelengths_rf)
    resazurin_spectra = resazurin_spectra.loc[common_wavelengths]
    resorufin_spectra = resorufin_spectra.loc[common_wavelengths]
    wavelengths = common_wavelengths

results = []
for pH in common_pH:
    abs_rz = resazurin_spectra[pH].to_numpy(dtype=float)
    abs_rf = resorufin_spectra[pH].to_numpy(dtype=float)
    for x in x_values:
        mixed_abs = mix_absorbance(wavelengths, abs_rz, abs_rf, x)
        reflectance = absorbance_to_reflectance(mixed_abs, S0)
        lab, rgb = spectrum_to_lab_rgb(wavelengths, reflectance)
        results.append({
            'pH': pH,
            'x': x,
            'L*': round(float(lab[0]), 3),
            'a*': round(float(lab[1]), 3),
            'b*': round(float(lab[2]), 3),
            'R': int(rgb[0]),
            'G': int(rgb[1]),
            'B': int(rgb[2]),
        })

results_df = pd.DataFrame(results)
print(f'Computed {len(results_df)} pH x x combinations.')
display(results_df.head(10))

Computed 187 pH x x combinations.


,pH,x,L*,a*,b*,R,G,B
0,1.0,0.0,77.777,33.439,15.123,255,168,166
1,1.0,0.1,78.251,32.041,16.173,255,170,165
2,1.0,0.2,78.768,30.645,17.283,255,173,164
3,1.0,0.3,79.324,29.206,18.443,255,175,164
4,1.0,0.4,79.921,27.709,19.660,255,178,163
5,1.0,0.5,80.566,26.139,20.943,255,181,162
6,1.0,0.6,81.265,24.481,22.305,255,184,162
7,1.0,0.7,82.030,22.717,23.761,255,187,161
8,1.0,0.8,82.874,20.822,25.337,255,191,160
9,1.0,0.9,83.819,18.765,27.068,255,195,159


In [7]:
#@title Build and download the Excel file with colored swatches
from datetime import datetime

OUTPUT_FILENAME = f"Resazurin_Resorufin_colors_{datetime.now().strftime('%Y%m%d_%H%M%S')}.xlsx"

export_df = results_df[['pH', 'x', 'L*', 'a*', 'b*']].copy()
export_df['Color'] = ''
export_df['R'] = results_df['R']
export_df['G'] = results_df['G']
export_df['B'] = results_df['B']

output_path = '/content/' + OUTPUT_FILENAME if IN_COLAB else OUTPUT_FILENAME

with pd.ExcelWriter(output_path, engine='xlsxwriter') as writer:
    export_df.to_excel(writer, sheet_name='Colors', index=False, startrow=0)
    workbook = writer.book
    worksheet = writer.sheets['Colors']

    header_format = workbook.add_format({'bold': True, 'align': 'center', 'valign': 'vcenter', 'border': 1})
    for col_idx, col_name in enumerate(export_df.columns):
        worksheet.write(0, col_idx, col_name, header_format)

    color_col = export_df.columns.get_loc('Color')
    for row_idx, (_, row) in enumerate(results_df.iterrows(), start=1):
        hex_color = '#{:02X}{:02X}{:02X}'.format(int(row['R']), int(row['G']), int(row['B']))
        cell_format = workbook.add_format({'bg_color': hex_color, 'border': 1})
        worksheet.write_blank(row_idx, color_col, None, cell_format)

    worksheet.set_column('A:A', 8)
    worksheet.set_column('B:B', 8)
    worksheet.set_column('C:E', 10)
    worksheet.set_column('F:F', 12)
    worksheet.set_column('G:I', 8)

print(f'Excel file created: {output_path}')

if IN_COLAB:
    files.download(output_path)

display(export_df.head(10))

Excel file created: /content/Resazurin_Resorufin_colors_20260811_120834.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,pH,x,L*,a*,b*,Color,R,G,B
0,1.0,0.0,77.777,33.439,15.123,,255,168,166
1,1.0,0.1,78.251,32.041,16.173,,255,170,165
2,1.0,0.2,78.768,30.645,17.283,,255,173,164
3,1.0,0.3,79.324,29.206,18.443,,255,175,164
4,1.0,0.4,79.921,27.709,19.660,,255,178,163
5,1.0,0.5,80.566,26.139,20.943,,255,181,162
6,1.0,0.6,81.265,24.481,22.305,,255,184,162
7,1.0,0.7,82.030,22.717,23.761,,255,187,161
8,1.0,0.8,82.874,20.822,25.337,,255,191,160
9,1.0,0.9,83.819,18.765,27.068,,255,195,159
